<a href="https://colab.research.google.com/github/Manase2008/ML-Projects/blob/main/Ensemble.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.ensemble import BaggingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score
import pandas as pd
from sklearn.preprocessing import LabelEncoder
encoder = LabelEncoder()
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()

data = pd.read_csv('adult.data.csv', header=None)
data = data.dropna(how='any')
data = data.drop(0, axis=1)
categorical_cols = [1, 3, 5, 6, 7, 8, 9, 13, 14]

# Apply LabelEncoder to these columns
for col in categorical_cols:
    if col in data.columns:
        data[col] = encoder.fit_transform(data[col])
X = data.drop([14], axis=1)
y = data[14]
X_scale = scaler.fit_transform(X)

# Train/Test split
X_train, X_test, y_train, y_test = train_test_split(
    X_scale, y,
    test_size=0.2,
    random_state=42
)

# learner
model = DecisionTreeClassifier(max_depth=13)

# Bagging ensemble with bootstrap sampling
bagging_model = BaggingClassifier(
    estimator= model,
    n_estimators=100,      # number of trees
    bootstrap=True,        # bootstrap sampling
    random_state=42,
    oob_score=True
)

# Train
bagging_model.fit(X_train, y_train)

# Predict
y_pred = bagging_model.predict(X_test)

# Evaluate
accuracy = accuracy_score(y_test, y_pred)
print(f"Ensemble Accuracy: {accuracy:.4f}")
print("OOB Score:", bagging_model.oob_score_)

Accuracy: 0.8604
OOB Score: 0.8556511056511057


In [ ]:
import numpy as np
models = bagging_model.estimators_

weights = np.ones(len(models))  # simple case
weights = weights / weights.sum()

In [ ]:
def weighted_predict_proba(models, weights, X):
    total = None

    for model, w in zip(models, weights):
        proba = model.predict_proba(X) * w

        if total is None:
            total = proba
        else:
            total += proba

    return total

In [ ]:
proba = weighted_predict_proba(models, weights, X_test)
y_pred = np.argmax(proba, axis=1)

accuracy = accuracy_score(y_test, y_pred)
print("Weighted Bagging Accuracy:", accuracy)

Weighted Bagging Accuracy: 0.8604329801934593
